In [0]:
%sql
CREATE OR REPLACE TABLE synchrony.analytics.customer_fy_sow AS
SELECT
    Customer_ID,
    Fiscal_Year,

    SUM(Total_MetroMart_Spend) AS Total_Spend,
    SUM(HSIC_Spend) AS HSIC_Spend,
    SUM(Other_Payment_Spend) AS Other_Payment_Spend,

    CASE
        WHEN SUM(Total_MetroMart_Spend) > 0
         AND SUM(HSIC_Spend) >= 0
         AND SUM(HSIC_Spend) <= SUM(Total_MetroMart_Spend)
        THEN SUM(HSIC_Spend) / SUM(Total_MetroMart_Spend)
        ELSE NULL
    END AS HSIC_SoW,

    SUM(HSIC_Transactions) AS HSIC_Transactions,
    SUM(Total_Transactions) AS Total_Transactions,

    CASE
        WHEN SUM(Total_MetroMart_Spend) <= 0
            THEN 'INVALID_NON_POSITIVE_SPEND'
        WHEN SUM(HSIC_Spend) < 0
          OR SUM(HSIC_Spend) > SUM(Total_MetroMart_Spend)
            THEN 'INVALID_SOW_RANGE'
        ELSE 'VALID'
    END AS SoW_Data_Quality

FROM synchrony.analytics.customer_monthly_sow
GROUP BY
    Customer_ID,
    Fiscal_Year;

In [0]:
%sql
select * from synchrony.analytics.customer_fy_sow

In [0]:
%sql
SELECT *
FROM synchrony.analytics.customer_fy_sow
ORDER BY Customer_ID, Fiscal_Year
LIMIT 50;

In [0]:
%sql
SELECT
    Fiscal_Year,
    SoW_Data_Quality,
    COUNT(*) AS Customer_Years,
    MIN(HSIC_SoW) AS Min_SoW,
    MAX(HSIC_SoW) AS Max_SoW
FROM synchrony.analytics.customer_fy_sow
GROUP BY
    Fiscal_Year,
    SoW_Data_Quality
ORDER BY
    Fiscal_Year,
    SoW_Data_Quality;